# Async DLCA on MNIST: WBC-NCA Update Methodology + Logic-Gate Classifier

This notebook is a build scaffold for a Differentiable Logic Cellular Automata (DLCA) classifier on MNIST. It borrows the update methodology from the WBC-NCA/WCA-style papers and adapts it to a logic-gate CA:

- each pixel is treated as a cell in a spatial grid;
- image/input channels are clamped back into the state after every CA step;
- each step samples a fresh per-cell stochastic update mask, so not every cell updates every iteration;
- the continuous NCA update MLP is replaced by differentiable logic perceive/update circuits;
- the classifier head receives thresholded popcount bits from hidden channels instead of a global max-pool.

The goal of this first notebook is smoke validation, not convergence on full MNIST. Full training, spatial-pyramid readouts, light-family gates, and hardware export are left as explicit next steps.


## Problem Framing

WBC-NCA uses a recurrent local update rule over an image grid. The image channels are preserved, hidden channels evolve, and a classifier reads the final hidden state. The stochastic update mask is not a chessboard mask: every cell independently gets an update/no-update sample at each CA step, and that one sample is broadcast across all channels of the cell.

For DLCA, the analogous scheme is:

1. seed a binary or soft-binary state from MNIST;
2. extract each cell's 3x3 Moore neighborhood;
3. run a differentiable-logic perception circuit over the patch;
4. run a differentiable-logic update circuit to propose the next full state for the center cell;
5. optionally apply a fresh per-cell async update mask;
6. reclamp the input channel(s);
7. summarize hidden channels with thresholded popcount bits;
8. classify those bits with another DLGN head.

The readout deliberately avoids global max-pooling. A binary global max is effectively an OR over space, so one stray active cell can turn an entire hidden channel into a `1`. Thresholded popcount preserves density: for each hidden channel, the classifier receives bits like "at least 1 / 4 / 16 / 64 cells were active." 


In [ ]:
# Imports and local package path
from typing import Any, NamedTuple, Union
import math
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'dlgn').exists():
    ROOT = Path('/Users/jaredhillyer/Documents/ALL_NCA_STUFF')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import jax
import jax.numpy as jnp
import jax.random as random
import numpy as np
import optax

from dlgn.models.conv import init_perceive_layer, run_perceive
from dlgn.models.initialization import init_gate_layer
from dlgn.models.network import run_logic_gate_network


In [ ]:
class TrainState(NamedTuple):
    params: Any
    opt_state: optax.OptState
    key: jax.Array
    step: int
    is_softmaxed: bool


class Batch(NamedTuple):
    x: jax.Array  # (B, H, W, C), NHWC
    y: jax.Array  # (B,), int labels


def make_config() -> dict[str, Any]:
    return {
        'seed': 0,
        'dataset': 'mnist',
        'num_classes': 10,
        'image_shape': (28, 28, 1),
        'input_channels': 1,
        'state_channels': 12,
        'hidden_channels': 11,
        'binarize_inputs': True,
        'input_threshold': 0.25,
        'num_steps': 3,
        'async_update': True,
        'update_rate': 0.6,
        'periodic': False,
        'logic_family': 'full',
        'architecture': 'softmax',
        'gumb_tau': 1.0,
        'dirichlet_concentration': 1.0,
        'perceive_channels': 24,
        'perceive_depth': 3,
        'perceive_connection': 'random',
        'update_layers': [64, 64],
        'update_connections': 'random',
        'readout_thresholds': [1.0, 4.0, 16.0, 64.0],
        'readout_temperature': 4.0,
        'classifier_layers': [64],
        'classifier_connections': 'random',
        'classifier_logit_scale': 5.0,
        'batch_size': 8,
        'lr': 1e-2,
        'clip_value': 1.0,
        'weight_decay': 1e-4,
        'smoke_batches': 3,
        'allow_mnist_download': False,
        'data_root': './data-mnist',
        'jit_train_step': False,
    }


config = make_config()
assert config['state_channels'] == config['input_channels'] + config['hidden_channels']
config


## Data Loading

The loader mirrors the MNIST handling from `conv_model_latest.py`: images become `float32` NHWC tensors. By default this cell does **not** download data. If MNIST is unavailable locally, it falls back to synthetic MNIST-shaped tensors so the model path can still be smoke-tested.


In [ ]:
def batch_to_jax(batch: Any) -> Batch:
    x, y = batch
    if hasattr(x, 'detach'):
        x = x.detach().cpu().numpy()
    else:
        x = np.asarray(x)

    if hasattr(y, 'detach'):
        y = y.detach().cpu().numpy()
    else:
        y = np.asarray(y)

    x = x.astype(np.float32)
    y = y.reshape(-1).astype(np.int32)

    if x.ndim == 2:
        x = x[None, ..., None]
    elif x.ndim == 3:
        x = x[..., None]
    elif x.ndim == 4 and x.shape[1] in (1, 3):
        x = np.transpose(x, (0, 2, 3, 1))
    elif x.ndim == 4 and x.shape[-1] in (1, 3):
        pass
    else:
        raise ValueError(f'Unexpected image batch shape: {x.shape}')

    return Batch(jnp.asarray(x, dtype=jnp.float32), jnp.asarray(y, dtype=jnp.int32))


def make_synthetic_mnist_batch(key: jax.Array, config: dict[str, Any]) -> Batch:
    key_x, key_y = random.split(key)
    batch_size = int(config['batch_size'])
    h, w, c = config['image_shape']
    # Sparse random strokes are enough for plumbing tests; they are not a training substitute.
    x = random.bernoulli(key_x, p=0.18, shape=(batch_size, h, w, c)).astype(jnp.float32)
    y = random.randint(key_y, shape=(batch_size,), minval=0, maxval=config['num_classes'])
    return Batch(x, y.astype(jnp.int32))


def load_mnist_or_synthetic(config: dict[str, Any], key: jax.Array) -> tuple[Batch, str]:
    try:
        import torch
        from torchvision import datasets, transforms

        transform = transforms.ToTensor()
        train_set = datasets.MNIST(
            root=config['data_root'],
            train=True,
            download=bool(config['allow_mnist_download']),
            transform=transform,
        )
        loader = torch.utils.data.DataLoader(
            train_set,
            batch_size=int(config['batch_size']),
            shuffle=True,
            drop_last=True,
            num_workers=0,
        )
        return batch_to_jax(next(iter(loader))), 'mnist-local'
    except Exception as exc:
        print('MNIST unavailable; using synthetic MNIST-shaped smoke data.')
        print('Reason:', repr(exc))
        return make_synthetic_mnist_batch(key, config), 'synthetic'


## Initialization

The parameter tree has three branches:

- `perceive`: shared DLGN tree kernels over a 3x3 patch;
- `update`: DLGN cell update circuit from center-state plus perceived features to the next full cell state;
- `classifier`: DLGN head over thresholded popcount readout bits.

The wires are kept separate from `TrainState`, matching the style of the DLCA notebook and the local `dlgn` package.


In [ ]:
def _connection_list(connections: Union[str, list[str]], n_layers: int) -> list[str]:
    if isinstance(connections, str):
        return [connections] * n_layers
    if len(connections) != n_layers:
        raise ValueError(f'Expected {n_layers} connection specs, got {len(connections)}')
    return list(connections)


def init_gate_stack(
    layer_sizes: list[int],
    connections: Union[str, list[str]],
    key: jax.Array,
    logic_family: str,
) -> tuple[list[jax.Array], list[tuple[jax.Array, jax.Array]]]:
    params = []
    wires = []
    connection_types = _connection_list(connections, len(layer_sizes) - 1)
    for in_dim, out_dim, connection_type in zip(layer_sizes[:-1], layer_sizes[1:], connection_types):
        key, subkey = random.split(key)
        layer_params, layer_wires = init_gate_layer(
            subkey,
            int(in_dim),
            int(out_dim),
            connection_type,
            logic_family,
        )
        params.append(layer_params)
        wires.append(layer_wires)
    return params, wires


def create_optimizer(config: dict[str, Any]) -> optax.GradientTransformation:
    return optax.chain(
        optax.clip(float(config['clip_value'])),
        optax.adamw(
            learning_rate=float(config['lr']),
            weight_decay=float(config['weight_decay']),
        ),
    )


def init_diff_logic_ca(
    config: dict[str, Any],
    key: jax.Array,
    opt: optax.GradientTransformation,
) -> tuple[TrainState, dict[str, Any]]:
    if config['logic_family'] != 'full':
        raise NotImplementedError('This v1 notebook is intentionally full-family only.')

    params: dict[str, Any] = {'perceive': None, 'update': None, 'classifier': None}
    wires: dict[str, Any] = {'perceive': None, 'update': None, 'classifier': None}

    key, perceive_key, update_key, classifier_key = random.split(key, 4)

    patch_dim = 9 * int(config['state_channels'])
    params['perceive'], wires['perceive'] = init_perceive_layer(
        perceive_key,
        patch_dim=patch_dim,
        n_kernels=int(config['perceive_channels']),
        depth=int(config['perceive_depth']),
        connection_type=config['perceive_connection'],
        logic_family=config['logic_family'],
    )

    update_input_dim = int(config['state_channels']) + int(config['perceive_channels'])
    update_sizes = [update_input_dim] + list(config['update_layers']) + [int(config['state_channels'])]
    params['update'], wires['update'] = init_gate_stack(
        update_sizes,
        config['update_connections'],
        update_key,
        config['logic_family'],
    )

    readout_dim = int(config['hidden_channels']) * len(config['readout_thresholds'])
    classifier_sizes = [readout_dim] + list(config['classifier_layers']) + [int(config['num_classes'])]
    params['classifier'], wires['classifier'] = init_gate_stack(
        classifier_sizes,
        config['classifier_connections'],
        classifier_key,
        config['logic_family'],
    )

    opt_state = opt.init(params)
    state = TrainState(
        params=params,
        opt_state=opt_state,
        key=key,
        step=0,
        is_softmaxed=(config['logic_family'] == 'full'),
    )
    return state, wires


## DLCA Rollout with WBC-NCA-Style Async Updates

The async mask has shape `(B, H, W, 1)`. One random decision controls whether the whole cell updates on that step; the decision is broadcast across all state channels. After the update, the MNIST input channel is clamped back into the state.


In [ ]:
def seed_mnist_state(images: jax.Array, config: dict[str, Any]) -> jax.Array:
    if images.ndim != 4:
        raise ValueError(f'Expected NHWC images, got {images.shape}')
    x = images.astype(jnp.float32)
    if config.get('binarize_inputs', True):
        x = (x >= float(config['input_threshold'])).astype(jnp.float32)

    batch, height, width, _ = x.shape
    state = jnp.zeros((batch, height, width, int(config['state_channels'])), dtype=jnp.float32)
    return state.at[..., : int(config['input_channels'])].set(x[..., : int(config['input_channels'])])


def reclamp_input_channels(state: jax.Array, seed_state: jax.Array, config: dict[str, Any]) -> jax.Array:
    input_channels = int(config['input_channels'])
    return state.at[..., :input_channels].set(seed_state[..., :input_channels])


def extract_moore_patches(state: jax.Array, periodic: bool) -> jax.Array:
    batch, height, width, channels = state.shape
    pad = ((0, 0), (1, 1), (1, 1), (0, 0))
    if periodic:
        padded = jnp.pad(state, pad, mode='wrap')
    else:
        padded = jnp.pad(state, pad, mode='constant', constant_values=0.0)

    patches = jax.lax.conv_general_dilated_patches(
        padded,
        filter_shape=(3, 3),
        window_strides=(1, 1),
        padding='VALID',
        dimension_numbers=('NHWC', 'HWIO', 'NHWC'),
    )
    return patches.reshape(batch * height * width, 9, channels)


def sample_update_mask(key: jax.Array, state_shape: tuple[int, ...], update_rate: float) -> jax.Array:
    mask_shape = state_shape[:-1] + (1,)
    return random.bernoulli(key, p=float(update_rate), shape=mask_shape).astype(jnp.float32)


def propose_next_state(
    state: jax.Array,
    params: dict[str, Any],
    wires: dict[str, Any],
    key: jax.Array,
    training: bool,
    config: dict[str, Any],
) -> jax.Array:
    batch, height, width, channels = state.shape
    key_perceive, key_update = random.split(key)

    patches = extract_moore_patches(state, periodic=bool(config['periodic']))
    perceived = run_perceive(
        params['perceive'],
        wires['perceive'],
        patches,
        training,
        key_perceive,
        architecture=config['architecture'],
        gumb_tau=float(config['gumb_tau']),
        dirichlet_concentration=float(config['dirichlet_concentration']),
        logic_family=config['logic_family'],
    )

    center_state = state.reshape(batch * height * width, channels)
    update_input = jnp.concatenate([center_state, perceived], axis=-1)
    proposed = run_logic_gate_network(
        params['update'],
        wires['update'],
        update_input,
        training,
        key_update,
        config['architecture'],
        float(config['gumb_tau']),
        float(config['dirichlet_concentration']),
        config['logic_family'],
    )
    return proposed.reshape(batch, height, width, channels)


def run_dlca_rollout(
    state: jax.Array,
    seed_state: jax.Array,
    params: dict[str, Any],
    wires: dict[str, Any],
    key: jax.Array,
    training: bool,
    config: dict[str, Any],
) -> jax.Array:
    num_steps = int(config['num_steps'])

    def body(carry, _):
        current_state, current_key = carry
        current_key, step_key, mask_key = random.split(current_key, 3)
        proposed = propose_next_state(current_state, params, wires, step_key, training, config)

        if bool(config['async_update']):
            mask = sample_update_mask(mask_key, proposed.shape, float(config['update_rate']))
            next_state = current_state * (1.0 - mask) + proposed * mask
        else:
            next_state = proposed

        next_state = reclamp_input_channels(next_state, seed_state, config)
        return (next_state, current_key), None

    (final_state, _), _ = jax.lax.scan(body, (state, key), xs=None, length=num_steps)
    return final_state


## Classifier Readout

The readout converts hidden-channel activity into bits. During training, each threshold is a sigmoid so gradients can flow. During hard evaluation, each threshold becomes a real bit.


In [ ]:
def readout_popcount_bits(state: jax.Array, training: bool, config: dict[str, Any]) -> jax.Array:
    input_channels = int(config['input_channels'])
    hidden = state[..., input_channels:]
    counts = jnp.sum(hidden, axis=(1, 2))
    thresholds = jnp.asarray(config['readout_thresholds'], dtype=state.dtype)

    if training:
        temperature = jnp.asarray(config['readout_temperature'], dtype=state.dtype)
        bits = jax.nn.sigmoid((counts[..., None] - thresholds) / temperature)
    else:
        bits = (counts[..., None] >= thresholds).astype(state.dtype)

    return bits.reshape(state.shape[0], -1)


def forward_logits(
    params: dict[str, Any],
    wires: dict[str, Any],
    images: jax.Array,
    key: jax.Array,
    training: bool,
    config: dict[str, Any],
) -> jax.Array:
    seed_state = seed_mnist_state(images, config)
    final_state = run_dlca_rollout(seed_state, seed_state, params, wires, key, training, config)
    readout = readout_popcount_bits(final_state, training, config)
    logits = run_logic_gate_network(
        params['classifier'],
        wires['classifier'],
        readout,
        training,
        key,
        config['architecture'],
        float(config['gumb_tau']),
        float(config['dirichlet_concentration']),
        config['logic_family'],
    )
    return float(config['classifier_logit_scale']) * logits


def loss_f(
    params: dict[str, Any],
    wires: dict[str, Any],
    batch: Batch,
    key: jax.Array,
    config: dict[str, Any],
) -> tuple[jax.Array, dict[str, jax.Array]]:
    key_soft, key_hard = random.split(key)
    logits_soft = forward_logits(params, wires, batch.x, key_soft, True, config)
    logits_hard = forward_logits(params, wires, batch.x, key_hard, False, config)

    soft_loss = optax.softmax_cross_entropy_with_integer_labels(logits_soft, batch.y).mean()
    hard_loss = optax.softmax_cross_entropy_with_integer_labels(logits_hard, batch.y).mean()
    soft_acc = (jnp.argmax(logits_soft, axis=-1) == batch.y).astype(jnp.float32).mean()
    hard_acc = (jnp.argmax(logits_hard, axis=-1) == batch.y).astype(jnp.float32).mean()

    return soft_loss, {
        'soft_loss': soft_loss,
        'hard_loss': hard_loss,
        'soft_acc': soft_acc,
        'hard_acc': hard_acc,
    }


def make_train_step(opt: optax.GradientTransformation, config: dict[str, Any]):
    def train_step(state: TrainState, wires: dict[str, Any], batch: Batch):
        key, loss_key = random.split(state.key)

        def apply_loss(params):
            return loss_f(params, wires, batch, loss_key, config)

        (loss, aux), grads = jax.value_and_grad(apply_loss, has_aux=True)(state.params)
        updates, opt_state = opt.update(grads, state.opt_state, state.params)
        params = optax.apply_updates(state.params, updates)
        new_state = state._replace(
            params=params,
            opt_state=opt_state,
            key=key,
            step=state.step + 1,
        )
        return new_state, loss, aux

    if bool(config.get('jit_train_step', False)):
        return jax.jit(train_step)
    return train_step


## Smoke Validation

This cell checks the intended contract before any real training effort:

- MNIST or synthetic batch is NHWC;
- seed state has the configured number of channels;
- async mask updates approximately `update_rate` of cells;
- input channels remain clamped after rollout;
- popcount readout has the expected dimensionality;
- classifier logits are `(batch, 10)`;
- a few optimizer steps produce finite metrics.


In [ ]:
key = random.PRNGKey(config['seed'])
key, init_key, data_key, smoke_key = random.split(key, 4)

opt = create_optimizer(config)
state, wires = init_diff_logic_ca(config, init_key, opt)
batch, data_source = load_mnist_or_synthetic(config, data_key)
print('data_source:', data_source)
print('batch.x:', batch.x.shape, batch.x.dtype)
print('batch.y:', batch.y.shape, batch.y.dtype)

expected_batch_shape = (config['batch_size'], *config['image_shape'])
assert batch.x.shape == expected_batch_shape, (batch.x.shape, expected_batch_shape)
assert batch.y.shape == (config['batch_size'],)

seed_state = seed_mnist_state(batch.x, config)
expected_state_shape = (
    config['batch_size'],
    config['image_shape'][0],
    config['image_shape'][1],
    config['state_channels'],
)
assert seed_state.shape == expected_state_shape, (seed_state.shape, expected_state_shape)

mask = sample_update_mask(smoke_key, seed_state.shape, config['update_rate'])
mask_fraction = float(mask.mean())
print('async mask fraction:', mask_fraction)
assert abs(mask_fraction - config['update_rate']) < 0.08

key, rollout_key = random.split(key)
rolled = run_dlca_rollout(seed_state, seed_state, state.params, wires, rollout_key, True, config)
input_channels = int(config['input_channels'])
clamp_error = float(jnp.max(jnp.abs(rolled[..., :input_channels] - seed_state[..., :input_channels])))
print('input clamp max error:', clamp_error)
assert clamp_error == 0.0

readout = readout_popcount_bits(rolled, True, config)
expected_readout_dim = config['hidden_channels'] * len(config['readout_thresholds'])
print('readout:', readout.shape)
assert readout.shape == (config['batch_size'], expected_readout_dim)

key, logits_key = random.split(key)
logits = forward_logits(state.params, wires, batch.x, logits_key, True, config)
print('logits:', logits.shape)
assert logits.shape == (config['batch_size'], config['num_classes'])
assert bool(jnp.all(jnp.isfinite(logits)))

train_step = make_train_step(opt, config)
for i in range(int(config['smoke_batches'])):
    state, loss, aux = train_step(state, wires, batch)
    assert bool(jnp.isfinite(loss))
    assert bool(jnp.isfinite(aux['hard_loss']))
    print(
        f"step={state.step} "
        f"soft_loss={float(aux['soft_loss']):.4f} "
        f"hard_loss={float(aux['hard_loss']):.4f} "
        f"soft_acc={float(aux['soft_acc']):.3f} "
        f"hard_acc={float(aux['hard_acc']):.3f}"
    )

print('Smoke validation complete.')


## Refinement Notes

Next iterations should be done in this order:

1. **Tune the smoke model into a real tiny-MNIST run**: increase `num_steps`, use real MNIST batches, and watch whether soft loss moves before touching the architecture.
2. **Compare readouts**: keep thresholded popcount as the baseline, then test spatial-pyramid popcount to preserve coarse location.
3. **Add light-family support**: reuse the light gate decoder/init path once the full-family path is stable.
4. **Add hard/soft divergence plots**: track whether hard discrete inference follows soft training or snaps to a different behavior.
5. **Export analysis**: inspect hard gate IDs in `perceive`, `update`, and `classifier` after training, then start thinking about hardware or Verilog paths.
